In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_core.prompts import ChatPromptTemplate

def user_input_formatter():
    system_prompt = """
You are an intent extraction expert.
Your task is only to identify the intent of user message, and to identify the social media platforms where user wants to post.

Platforms to identify
Linkedin
Twitter
Instagram

Rules:
-If a platform is explicitly mentioned, set its value to true.
-If a platform is not mentioned, set its value to false.
- If multiple platform is mentioned, set each of them to true.
- Only detect the intent related to posting on these platforms.
- Do no generate explaination.

Return ONLY valid JSON:
{{
    "linkedin":boolean,
    "twitter":boolean,
    "instagram":boolean,
    "user_message":"user message with platform names removed"
}}

"If the user message does not mention any platform return all values to false"
"""
    return  ChatPromptTemplate([
        ('system',system_prompt),
        ('user',"{input_query}")
    ])

In [3]:
from langchain.chat_models import init_chat_model
def input_intent_analyst():
    return init_chat_model("groq:llama-3.1-8b-instant", temperature=0)

In [4]:
chain =  user_input_formatter() | input_intent_analyst() 
chain.invoke({
    "input_query": "I want to post about India 2026 world cup win in linkedin and twitter"
})

AIMessage(content='{\n    "linkedin": true,\n    "twitter": true,\n    "instagram": false,\n    "user_message": "I want to post about India 2026 world cup win"\n}', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 210, 'total_tokens': 250, 'completion_time': 0.138625652, 'completion_tokens_details': None, 'prompt_time': 0.013102776, 'prompt_tokens_details': None, 'queue_time': 0.005265927, 'total_time': 0.151728428}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d15ee-45f4-78a0-a2fc-a6d7f655cc14-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 210, 'output_tokens': 40, 'total_tokens': 250})

In [5]:
user_input = input("Enter the request:")
chain =  user_input_formatter() | input_intent_analyst() 
chain.invoke({"input_query":user_input})

AIMessage(content='{\n    "linkedin":false,\n    "twitter":false,\n    "instagram":false,\n    "user_message":"fff"\n}', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 194, 'total_tokens': 222, 'completion_time': 0.032219852, 'completion_tokens_details': None, 'prompt_time': 0.026854679, 'prompt_tokens_details': None, 'queue_time': 0.017240752, 'total_time': 0.059074531}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d15ee-4ec0-7753-b0d5-a76c7ec83da6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 194, 'output_tokens': 28, 'total_tokens': 222})

In [6]:
def input_node(input_query:str):
    chain = user_input_formatter() | input_intent_analyst()
    return chain.invoke({"input_query":input_query})

# Langraph Module - Parallel Orchestration

In [7]:
from typing import List, TypedDict, Annotated
from operator import add, or_
from langgraph.graph import START, END, StateGraph
class InputSchema(TypedDict):
    platform_checker: Annotated[List, add]
    input_query: str
    messages: Annotated[List, add]
    tool_messages: Annotated[List, add]
    search_counts: Annotated[dict, or_]

class OutputSchema(TypedDict):
    final_message: Annotated[List, add]


In [8]:
import re
import json
def platform_detector_node(state:InputSchema):
   input_state = state
   input_query = state["input_query"]
   response = input_node(input_query)
   response_content = response.content.strip()
   response_content = re.sub(r"^```json\s*|\s*```$", "", response_content, flags=re.MULTILINE)
   data = json.loads(response_content)
   platforms = [{
      "linkedin": data["linkedin"],
      "instagram": data["instagram"],
      "twitter": data["twitter"]
   }]
   input_state["platform_checker"] = platforms
   input_state["messages"] = [{"platform_detector_response":data["user_message"]}]
   return input_state

In [9]:
def platform_router(state:InputSchema):
    platforms = state["platform_checker"][0]
    return [f"{p}_executor_node" for p,status in platforms.items() if status]

In [10]:
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper,WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchResults,WikipediaQueryRun
from langchain.tools import tool

@tool
def web_search(query:str):
    """Use this to search latest news on topic requested by user"""
    wrapper = DuckDuckGoSearchAPIWrapper(region="de-de", time="d", max_results=2)
    search = DuckDuckGoSearchResults(api_wrapper=wrapper, source="news")
    return search.invoke(query)

@tool
def search_wiki(query:str):
    """Used for Wiki Search, """
    wrapper = WikipediaAPIWrapper()
    wikipedia_query = WikipediaQueryRun(api_wapper=wrapper)
    return wikipedia_query.invoke(query)


In [11]:
from langchain.agents import create_agent
from langgraph.types import Command,Send
from typing import Literal
def linkedin_executor_node(state:InputSchema)->Command[Literal["reducer_node"]]:
    input_state = state
    messages = state["messages"]
    human_msg = next((m for m in messages if 'platform_detector_response' in m),None)
    system_msg = """
        You are a professional LinkedIn content writer. 
        Create a detailed and insightful LinkedIn post about the topic requested by the user. 
        Use a professional tone, include context based on current and historical facts, and structure the post in clear paragraphs suitable for LinkedIn audiences.
    """
    linkedin_agent = create_agent(
        model="groq:llama-3.1-8b-instant",
        tools = [search_wiki, web_search],
        system_prompt=system_msg,
        
    )
    response = linkedin_agent.invoke({
        "messages": [
            {"role": "user", "content": f"Write a 100-line post about {human_msg}"}
        ]
    })
    final_message_obj = response["messages"][-1]
    final_content = final_message_obj.content
    print("\n--- FINAL LINKEDIN POST ---")
    print(final_content)
    print("------------------------------\n")
    if(final_content):
        return Command( update={"final_message": [{"linkedin_final_response": final_content}]},
             goto=["reducer_node"])

        




In [12]:
def post_prompt_handler():
    return ChatPromptTemplate({
        ('system',"{scope}"),
        ('user',"{human_message}")
    })

def post_generator_builder(scope:str,human_message:str,_tools:None):
    if _tools is None:
        _tools = [web_search]
    llm = input_intent_analyst()
    llm_tools = llm.bind_tools(_tools)
    prompt = post_prompt_handler()
    chain = prompt | llm_tools
    return chain.invoke({
        "scope":scope,
        "human_message":human_message
    })

In [13]:
def twitter_executor_node(state:InputSchema)->Command[Literal["reducer_node"]]:
    input_state = state
    messages = state["messages"]
    # human_msg = next((m for m in messages if 'platform_detector_response' in m),None)
    system_msg = """
    You are a professional humor writer.
    
    PHASE 1: RESEARCH
    Check if the user's topic requires current facts. If yes, use the ONLY available tool: 'web_search'.
    DO NOT attempt to use any other tools. 
    
    PHASE 2: WRITE
    Once you have the facts (or if none are needed), write a comical one-liner.
    Tone: Witty and sharp.
    Format: Single string for Twitter.
    """
    _search_counts = state["search_counts"] or {}
    twitter_count = _search_counts.get("twitter", 0)
    if twitter_count > 2:
        system_msg += """
            
            IMPORTANT: You have reached your search limit (2/2). 
            DO NOT use tools again
            Use the existing ToolMessage results and twitter_ai_response in the conversation history to write your final response now.
        """
    available_tools = [web_search] if twitter_count < 2 else []
    response = post_generator_builder(system_msg,messages,available_tools)
    if(response.content):
        return Command( update={"final_message": [{"twitter_final_response": response}]},
            goto=["reducer_node"])
    else:
        state["messages"] = [{"twitter_ai_response":response}]
        return Send("tool_node",[{"current_platform":"twitter"}, {"messages":state["messages"], "search_counts":state["search_counts"]}])

def instagram_executor_node(state:InputSchema)->Command[Literal["reducer_node"]]:
    input_state = state
    messages = state["messages"]
    # human_msg = next((m for m in messages if 'platform_detector_response' in m),None)
    system_msg = """
        You are a professional writer. 
        Create a post about the topic requested by the user. If yes, use the ONLY available tool: 'web_search'.
        DO NOT attempt to use any other tools. 
        Use a neutral tone, include context based on current facts, and structure the post in two liner for Instagram audiences.
    """
    _search_counts = state["search_counts"] or {}
    instagram_count = _search_counts.get("instagram", 0)
    if instagram_count > 2:
        system_msg += """
            
            IMPORTANT: You have reached your search limit (2/2). 
            DO NOT use tools again
            Use the existing ToolMessage results and instagram_ai_response in the conversation history to write your final response now.
        """
    available_tools = [web_search] if instagram_count < 2 else []
    response = post_generator_builder(system_msg,messages,available_tools)
    if(response.content):
        return Command( update={"final_message": [{"instagram_final_response": response}]},
            goto=["reducer_node"])
    else:
        state["messages"] = [{"instagram_ai_response":response}]
        return Send("tool_node",[{"current_platform":"instagram"}, {"messages":state["messages"],  "search_counts":state["search_counts"]}])

In [14]:
from langchain_core.messages import ToolMessage
def tool_node(state:InputSchema)->Command[Literal["twitter_executor_node","instagram_executor_node"]]:
    messages = next((d["messages"] for d in state if 'messages' in d),None)
    platform = next((d["current_platform"] for d in state if 'current_platform' in d),None)
    p_name = f"{platform}_ai_response"
    tool_result = []
    _tools = [web_search]
    toolnames = {tool.name: tool for tool in _tools}
    _search_counts = next((d["search_counts"] for d in state if 'search_counts' in d),{platform:0})
    platform_count = _search_counts.get(platform, 0)
    ai_msg_dict = next((m for m in messages if p_name in m),{})
    ai_msg = ai_msg_dict.get(p_name)
    if ai_msg and hasattr(ai_msg, "tool_calls"):
        for tool_call in ai_msg.tool_calls:
            tool = toolnames[tool_call["name"]]
            tool_args = tool_call["args"]
            if platform_count < 2:
                tool = toolnames[tool_call["name"]]
                tool_args = tool_call["args"]
                observation = tool.invoke(tool_args)
                platform_count += 1
                print(observation)
            else:
                observation = f"LIMIT Reached for {platform} has no searches left, Write the post now"
            tool_result.append(
                (ToolMessage(content=observation, tool_call_id=tool_call["id"]))
            )
    if platform == 'twitter':
        return Command(
            update={"messages":messages+tool_result, "search_counts":{platform: platform_count}}, goto=["twitter_executor_node"]
        )
    if platform == 'instagram':
        return Command(
            update={"messages":messages+tool_result, "search_counts":{platform: platform_count}}, goto=["instagram_executor_node"]
        )
        
    

In [15]:

def reducer_node(state:InputSchema):
    all_posts = state.get("final_message", [])  
    print(f"--- Finalizing {all_posts} posts ---")
    return {"final_message": ["Successfully aggregated all platform posts."]}

In [17]:

graph = StateGraph(InputSchema,input_schema=InputSchema, output_schema=OutputSchema)
graph.add_node("platform_detector_node", platform_detector_node)
graph.add_node("linkedin_executor_node", linkedin_executor_node)
graph.add_node("twitter_executor_node", twitter_executor_node)
graph.add_node("instagram_executor_node", instagram_executor_node)
graph.add_node("reducer_node",reducer_node)
graph.add_node("tool_node",tool_node)

graph.add_edge(START,"platform_detector_node")
graph.add_conditional_edges("platform_detector_node",platform_router)
graph.add_edge("platform_detector_node",END)
# graph.add_edge("platform_detector_node","linkedin_executor_node")
# graph.add_edge("linkedin_executor_node","twitter_executor_node")
# graph.add_edge("twitter_executor_node","instagram_executor_node")
# graph.add_edge("instagram_executor_node","reducer_node")
# graph.add_edge("reducer_node",END)

graph_constructor = graph.compile()
graph_constructor.invoke({
   "input_query":"I need to write a content about India team in cricket World Cup to post in linkedin , instagram and twitter "
})


--- FINAL LINKEDIN POST ---
The Indian cricket team has a rich history in the Cricket World Cup, with a total of two titles won in 1983 and 2011. The team has consistently been one of the top contenders in the tournament, with a strong lineup of players.

In the 1983 Cricket World Cup, India, led by Kapil Dev, defeated the West Indies in the final, marking a historic victory for the team. This win was a significant milestone for Indian cricket, as it marked the country's first major international victory.

In the 2011 Cricket World Cup, India, led by MS Dhoni, defeated Sri Lanka in the final, winning the tournament by 6 wickets. This win was a testament to the team's strength and determination, as they fought hard to overcome the challenges posed by their opponents.

The Indian cricket team has also had its share of successes in the World Cup, with a number of players making significant contributions to the team's victories. Players like Sachin Tendulkar, Sourav Ganguly, and Virender 

{'final_message': [{'linkedin_final_response': "The Indian cricket team has a rich history in the Cricket World Cup, with a total of two titles won in 1983 and 2011. The team has consistently been one of the top contenders in the tournament, with a strong lineup of players.\n\nIn the 1983 Cricket World Cup, India, led by Kapil Dev, defeated the West Indies in the final, marking a historic victory for the team. This win was a significant milestone for Indian cricket, as it marked the country's first major international victory.\n\nIn the 2011 Cricket World Cup, India, led by MS Dhoni, defeated Sri Lanka in the final, winning the tournament by 6 wickets. This win was a testament to the team's strength and determination, as they fought hard to overcome the challenges posed by their opponents.\n\nThe Indian cricket team has also had its share of successes in the World Cup, with a number of players making significant contributions to the team's victories. Players like Sachin Tendulkar, Sour